# Phase 4 — Geospatial Python: The Concepts Underneath Your Project (Addis Ababa Edition)

You've already run GeoPandas, OSMnx, and Folium in a real project (`addis-transit-access`). This notebook doesn't teach you new syntax you haven't seen — it explains **why** that syntax works, so the code stops being "pattern I copied" and becomes something you could rebuild from memory.

**Sections:**
1. What a CRS actually is, geometrically
2. Geometry types: Point, LineString, Polygon (Shapely basics)
3. GeoDataFrames — a DataFrame with one geometry column
4. Reprojection — why EPSG:4326 vs EPSG:32637 matters for real calculations
5. Spatial predicates — the logic behind "Select by Location" in QGIS
6. Buffers, unions, dissolves — what's mathematically happening
7. Spatial joins — combining attribute data with geometry
8. Raster basics — vectors vs. rasters, conceptually
9. Checkpoint — rebuild your project's core accessibility logic from scratch


## 1. What a CRS actually is, geometrically

A **Coordinate Reference System (CRS)** answers one question: "when I say a point is at (9.02, 38.75), what does that number actually mean on the Earth's surface?"

There are two fundamentally different kinds:

- **Geographic CRS** (e.g. `EPSG:4326`, WGS84) — coordinates are **degrees of latitude/longitude** on a curved model of the Earth (an ellipsoid). Good for storing/sharing data globally. **Bad for measuring distance or area directly** — a degree of longitude is a different real-world distance near the equator vs. near the poles.
- **Projected CRS** (e.g. `EPSG:32637`, UTM Zone 37N — covers Addis Ababa) — coordinates are in **meters**, on a flat mathematical projection of a specific region of the Earth. Good for accurate distance/area math. **Bad for global data** — the projection distorts badly far from its intended zone.

**This is why your transit-access project reprojects to `EPSG:32637` before buffering** — buffering "500 meters" only means something real in a projected CRS. Buffering in `EPSG:4326` would create a "500 degree" buffer, which is nonsensical (500 degrees doesn't even fit on Earth) — in practice you'd get badly distorted results, not a clean error, which is the dangerous part.

In [1]:
from shapely.geometry import Point
import geopandas as gpd
import pyproj

# A single point representing central Addis Ababa, in WGS84 (lat/lon degrees)
addis_center = Point(38.7525, 9.0192)  # NOTE: Shapely takes (x, y) = (lon, lat), not (lat, lon)!

gdf = gpd.GeoDataFrame({"name": ["Addis Ababa center"]}, geometry=[addis_center], crs="EPSG:4326")
print(gdf)
print("CRS:", gdf.crs)
print("Is this CRS geographic (degrees)?", gdf.crs.is_geographic)

                 name                geometry
0  Addis Ababa center  POINT (38.7525 9.0192)
CRS: EPSG:4326
Is this CRS geographic (degrees)? True


### Exercise 1.1
Create a `GeoDataFrame` with a single `Point` representing Meskel Square, Addis Ababa, at approximately latitude `9.0105`, longitude `38.7613`. Set its CRS to `EPSG:4326`. Then print whether that CRS is geographic or projected using `.crs.is_geographic`.

**Careful:** remember Shapely's `Point()` takes `(longitude, latitude)` order — the opposite of how humans usually say coordinates. This trips up almost everyone at least once.

In [ ]:
# Your code here
meskel_square = Point(38.7613, 9.0105)  # (lon, lat)

meskel_gdf = gpd.GeoDataFrame({"name": ["Meskel Square"]}, geometry=[meskel_square], crs="EPSG:4326")
print(meskel_gdf)
print("Is geographic?", meskel_gdf.crs.is_geographic)

: 

<details>
<summary>Solution 1.1</summary>

```python
meskel_square = Point(38.7613, 9.0105)  # (lon, lat)

meskel_gdf = gpd.GeoDataFrame({"name": ["Meskel Square"]}, geometry=[meskel_square], crs="EPSG:4326")
print(meskel_gdf)
print("Is geographic?", meskel_gdf.crs.is_geographic)
```
</details>

## 2. Geometry types — Point, LineString, Polygon

Shapely is the library underneath GeoPandas that represents shapes. Three types cover almost everything you'll need:

- **`Point`** — a single location (a bus stop, a building)
- **`LineString`** — a connected sequence of points (a road, a bus route)
- **`Polygon`** — a closed shape with an interior (a sub-city boundary)

Each has useful built-in properties: `.length` (for lines), `.area` (for polygons), `.centroid` (for any of them).

In [3]:
from shapely.geometry import Point, LineString, Polygon

# A bus stop
stop = Point(38.7525, 9.0192)

# A short bus route (simplified, 3 points)
route = LineString([(38.7525, 9.0192), (38.7550, 9.0210), (38.7580, 9.0230)])

# A rough rectangular "sub-city" boundary
boundary = Polygon([
    (38.74, 9.00), (38.77, 9.00), (38.77, 9.03), (38.74, 9.03), (38.74, 9.00)
])

print("Route length (in CRS units, degrees here — not meaningful without reprojecting!):", route.length)
print("Boundary area (same caveat):", boundary.area)
print("Boundary centroid:", boundary.centroid)

Route length (in CRS units, degrees here — not meaningful without reprojecting!): 0.006686135635617926
Boundary area (same caveat): 0.0009000000000000149
Boundary centroid: POINT (38.755 9.015)


### Exercise 2.1
Create a `Polygon` representing a rough triangular area using any 3 points near Addis Ababa (remember to close the ring by repeating the first point at the end, or Shapely does this for you with `Polygon(coords)` as long as you pass a valid ring). Print its `.centroid` and `.area`.

Then create a `Point` for any location and check whether it falls `.within()` your polygon.

In [20]:
# Your code here

triangle = Polygon([(38.74, 9.00), (38.77, 9.00), (38.755, 9.03)])
print("Centroid:", triangle.centroid)
print("Area:", triangle.area)

test_point = Point(38.755, 9.01)
print("Is point within triangle?", test_point.within(triangle))

Centroid: POINT (38.755 9.010000000000002)
Area: 0.00045000000000000747
Is point within triangle? True


<details>
<summary>Solution 2.1</summary>

```python
triangle = Polygon([(38.74, 9.00), (38.77, 9.00), (38.755, 9.03)])
print("Centroid:", triangle.centroid)
print("Area:", triangle.area)

test_point = Point(38.755, 9.01)
print("Is point within triangle?", test_point.within(triangle))
```
</details>

## 3. GeoDataFrames — a DataFrame with one geometry column

This is worth stating plainly since it demystifies a lot: a `GeoDataFrame` **is** a Pandas `DataFrame`. Every single thing you learned in Phase 2-3 (`.loc`, `groupby`, `merge`, `apply`) still works exactly the same. The only difference is one special column (usually named `geometry`) holds Shapely objects instead of numbers/text, and that column powers spatial operations.

In [4]:
subcity_polygons = gpd.GeoDataFrame({
    "sub_city": ["Bole", "Yeka"],
    "population": [328900, 401000],
    "geometry": [
        Polygon([(38.74, 8.98), (38.82, 8.98), (38.82, 9.05), (38.74, 9.05)]),
        Polygon([(38.80, 9.00), (38.88, 9.00), (38.88, 9.08), (38.80, 9.08)])
    ]
}, crs="EPSG:4326")

# Ordinary Pandas operations still work fine
print(subcity_polygons.groupby("sub_city")["population"].sum())
print()
# Plus geometry-specific operations
print(subcity_polygons.geometry.area)

sub_city
Bole    328900
Yeka    401000
Name: population, dtype: int64



C:\Users\sher khan\AppData\Local\Temp\ipykernel_1832\1484324576.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  print(subcity_polygons.geometry.area)


0    0.0056
1    0.0064
dtype: float64


### Exercise 3.1
Using `subcity_polygons` above, add a new column `pop_density` = `population / geometry.area` (yes, this is geometrically meaningless in degrees — that's the point, see Section 4 next). Then use ordinary Pandas `.sort_values()` to find which sub-city has the higher raw ratio.

**Heads up:** GeoPandas will actually print a `UserWarning` when you run this — something like *"Geometry is in a geographic CRS. Results from 'area' are likely incorrect."* That's not a bug in your code; it's GeoPandas itself telling you exactly what Section 4 is about to explain. Don't suppress it — read it.

In [25]:
# Your code here
subcity_polygons["pop_density"] = subcity_polygons["population"] / subcity_polygons.geometry.area
print(subcity_polygons.sort_values("pop_density", ascending=False)[["sub_city", "pop_density"]])

C:\Users\sher khan\AppData\Local\Temp\ipykernel_14200\360120922.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subcity_polygons["pop_density"] = subcity_polygons["population"] / subcity_polygons.geometry.area


  sub_city   pop_density
1     Yeka  6.265625e+07
0     Bole  5.873214e+07


<details>
<summary>Solution 3.1</summary>

```python
subcity_polygons["pop_density"] = subcity_polygons["population"] / subcity_polygons.geometry.area
print(subcity_polygons.sort_values("pop_density", ascending=False)[["sub_city", "pop_density"]])
```

This ran fine — Pandas doesn't know or care that the units are meaningless. That's exactly why Section 4 (reprojection) matters: GeoPandas will happily give you wrong-but-confident numbers if you forget to reproject first.
</details>

## 4. Reprojection — `EPSG:4326` vs `EPSG:32637`

`.to_crs()` converts a GeoDataFrame's coordinates from one CRS to another. Your project uses this exact pattern: fetch data in `EPSG:4326` (how OSM/most sources provide it) → reproject to `EPSG:32637` (UTM 37N, meters, correct for Addis Ababa's location) → do area/distance calculations → optionally reproject back to `EPSG:4326` or `EPSG:3857` for mapping with Folium/contextily.

In [5]:
# Reproject subcity_polygons to UTM 37N (meters)
subcity_utm = subcity_polygons.to_crs("EPSG:32637")

print("Original area (degrees^2, meaningless):", subcity_polygons.geometry.area.tolist())
print("Reprojected area (m^2, meaningful):", subcity_utm.geometry.area.tolist())
print()
print("Reprojected area in km^2:", (subcity_utm.geometry.area / 1_000_000).tolist())

Original area (degrees^2, meaningless): [0.005599999999999904, 0.0064000000000004375]
Reprojected area (m^2, meaningful): [68048234.18675694, 77763625.53088138]

Reprojected area in km^2: [68.04823418675694, 77.76362553088137]


C:\Users\sher khan\AppData\Local\Temp\ipykernel_1832\2149501191.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  print("Original area (degrees^2, meaningless):", subcity_polygons.geometry.area.tolist())


### Exercise 4.1
Take `subcity_polygons` from Section 3, reproject it to `EPSG:32637`, and compute the population density properly this time — population divided by area **in km²** (not raw m², which would give a tiny, hard-to-read number). Compare this to the meaningless Section 3 numbers.

In [ ]:
# Your code here


<details>
<summary>Solution 4.1</summary>

```python
subcity_utm2 = subcity_polygons.to_crs("EPSG:32637")
area_km2 = subcity_utm2.geometry.area / 1_000_000

subcity_utm2["pop_density_correct"] = subcity_utm2["population"] / area_km2
print(subcity_utm2[["sub_city", "pop_density_correct"]])
```

Notice these numbers are now in a sensible range (people per km²) — the Section 3 numbers were technically "computed" but geometrically meaningless, which is the dangerous part: nothing crashed, it just quietly lied.
</details>

## 5. Spatial predicates — the logic behind "Select by Location"

These are the actual functions QGIS's "Select by Location" tool calls under the hood. The main ones:

- **`.within(other)`** — this geometry is entirely inside `other`
- **`.contains(other)`** — this geometry entirely contains `other` (the reverse of `within`)
- **`.intersects(other)`** — they share at least one point (overlapping, touching, or one inside the other — the loosest/most common check)
- **`.touches(other)`** — they share a boundary but don't overlap internally

**For your work:** "which transit stops fall within this sub-city boundary?" is a `.within()` check. "Which sub-cities does this bus route pass through?" is usually `.intersects()`, since a route might just clip the edge of a sub-city without a stop being fully inside it.

In [6]:
bole_boundary = subcity_polygons.iloc[0].geometry
some_stop = Point(38.76, 9.00)   # a point we're checking

print("Is the stop within Bole's boundary?", some_stop.within(bole_boundary))
print("Does Bole's boundary contain the stop?", bole_boundary.contains(some_stop))  # same question, reversed direction
print("Do they intersect?", some_stop.intersects(bole_boundary))

Is the stop within Bole's boundary? True
Does Bole's boundary contain the stop? True
Do they intersect? True


### Exercise 5.1
Create 3 `Point`s representing hypothetical bus stops (pick coordinates yourself, some inside `bole_boundary` and some outside). Loop through them and print which ones are `.within()` Bole's boundary.

In [9]:
# Your code here
candidate_stops = [
    Point(38.76, 9.00),   # likely inside
    Point(38.90, 9.10),   # likely outside
    Point(38.78, 9.02)    # likely inside
]

for i, stop in enumerate(candidate_stops):
    print(f"Stop {i}: within Bole? {stop.within(bole_boundary)}")

Stop 0: within Bole? True
Stop 1: within Bole? False
Stop 2: within Bole? True


<details>
<summary>Solution 5.1</summary>

```python
candidate_stops = [
    Point(38.76, 9.00),   # likely inside
    Point(38.90, 9.10),   # likely outside
    Point(38.78, 9.02)    # likely inside
]

for i, stop in enumerate(candidate_stops):
    print(f"Stop {i}: within Bole? {stop.within(bole_boundary)}")
```
</details>

## 6. Buffers, unions, dissolves — what's mathematically happening

- **`.buffer(distance)`** — creates a new polygon expanded outward by `distance` in the CRS's units (meters, if you're in a projected CRS — another reason reprojecting first matters). Every point on the original geometry gets pushed outward equally, like inflating a shape.
- **`.union()`** — merges multiple geometries into one, dissolving any shared boundary
- **`.dissolve(by="column")`** in GeoPandas — groups rows by a column and unions their geometries together (e.g., merging all sub-city polygons that belong to one broader zone)

In [7]:
# Buffer a stop by 500 meters (must be in a projected CRS for this to mean meters!)
stop_utm = gpd.GeoSeries([Point(38.7525, 9.0192)], crs="EPSG:4326").to_crs("EPSG:32637")
buffered = stop_utm.buffer(500)

print("Buffer area in km^2:", (buffered.area / 1_000_000).round(3).tolist())
# A perfect 500m-radius circle has area = pi * 500^2 = ~785,398 m^2 ≈ 0.785 km^2 — check this matches roughly

Buffer area in km^2: [0.784]


### Exercise 6.1
Create two overlapping circular buffers (from two different points, each buffered 300m, in `EPSG:32637`) and use `.union()` to merge them into a single combined coverage area. Print the combined area in km² and compare it to the sum of the two individual buffer areas — it should be **less** than the sum, since they overlap.

In [10]:
# Your code here
points_utm = gpd.GeoSeries(
    [Point(38.7525, 9.0192), Point(38.7540, 9.0200)],
    crs="EPSG:4326"
).to_crs("EPSG:32637")

buffer1 = points_utm.iloc[0].buffer(300)
buffer2 = points_utm.iloc[1].buffer(300)

combined = buffer1.union(buffer2)

sum_individual = (buffer1.area + buffer2.area) / 1_000_000
combined_area = combined.area / 1_000_000

print(f"Sum of individual buffers: {sum_individual:.3f} km^2")
print(f"Combined (unioned) area: {combined_area:.3f} km^2")
print(f"Overlap accounted for: {sum_individual - combined_area:.3f} km^2")

Sum of individual buffers: 0.565 km^2
Combined (unioned) area: 0.393 km^2
Overlap accounted for: 0.172 km^2


<details>
<summary>Solution 6.1</summary>

```python
points_utm = gpd.GeoSeries(
    [Point(38.7525, 9.0192), Point(38.7540, 9.0200)],
    crs="EPSG:4326"
).to_crs("EPSG:32637")

buffer1 = points_utm.iloc[0].buffer(300)
buffer2 = points_utm.iloc[1].buffer(300)

combined = buffer1.union(buffer2)

sum_individual = (buffer1.area + buffer2.area) / 1_000_000
combined_area = combined.area / 1_000_000

print(f"Sum of individual buffers: {sum_individual:.3f} km^2")
print(f"Combined (unioned) area: {combined_area:.3f} km^2")
print(f"Overlap accounted for: {sum_individual - combined_area:.3f} km^2")
```
</details>

## 7. Spatial joins — combining attribute data with geometry

`gpd.sjoin()` is like Pandas `merge()`, but the "key" is spatial relationship instead of a shared column value. This is the core operation behind "which sub-city does each transit stop belong to?" — exactly what your accessibility project needs.

In [8]:
stops_gdf = gpd.GeoDataFrame({
    "stop_name": ["Stop A", "Stop B", "Stop C"],
    "geometry": [Point(38.76, 9.00), Point(38.84, 9.04), Point(38.78, 9.02)]
}, crs="EPSG:4326")

# Spatial join: for each stop, find which sub-city polygon it falls within
joined = gpd.sjoin(stops_gdf, subcity_polygons, how="left", predicate="within")
print(joined[["stop_name", "sub_city", "population"]])

  stop_name sub_city  population
0    Stop A     Bole      328900
1    Stop B     Yeka      401000
2    Stop C     Bole      328900


### Exercise 7.1
Notice `how="left"` was used above — same reasoning as Pandas merges from Phase 2. Add one more stop to `stops_gdf` at a location that falls **outside both** sub-city polygons (e.g., far away coordinates), redo the spatial join, and observe what happens to that row. Explain in a comment why `left` (not `inner`) is the right choice here too.

In [11]:
# Your code here
stops_gdf2 = gpd.GeoDataFrame({
    "stop_name": ["Stop A", "Stop B", "Stop C", "Stop D (far away)"],
    "geometry": [Point(38.76, 9.00), Point(38.84, 9.04), Point(38.78, 9.02), Point(39.50, 9.50)]
}, crs="EPSG:4326")

joined2 = gpd.sjoin(stops_gdf2, subcity_polygons, how="left", predicate="within")
print(joined2[["stop_name", "sub_city", "population"]])

# Stop D gets NaN for sub_city/population — it didn't fall within any polygon.
# A LEFT join keeps it visible (as an unmatched stop worth investigating —
# maybe it's genuinely outside city limits, or maybe the boundary data is incomplete).
# An INNER join would silently drop it, hiding a potential data quality issue.

           stop_name sub_city  population
0             Stop A     Bole    328900.0
1             Stop B     Yeka    401000.0
2             Stop C     Bole    328900.0
3  Stop D (far away)      NaN         NaN


<details>
<summary>Solution 7.1</summary>

```python
stops_gdf2 = gpd.GeoDataFrame({
    "stop_name": ["Stop A", "Stop B", "Stop C", "Stop D (far away)"],
    "geometry": [Point(38.76, 9.00), Point(38.84, 9.04), Point(38.78, 9.02), Point(39.50, 9.50)]
}, crs="EPSG:4326")

joined2 = gpd.sjoin(stops_gdf2, subcity_polygons, how="left", predicate="within")
print(joined2[["stop_name", "sub_city", "population"]])

# Stop D gets NaN for sub_city/population — it didn't fall within any polygon.
# A LEFT join keeps it visible (as an unmatched stop worth investigating —
# maybe it's genuinely outside city limits, or maybe the boundary data is incomplete).
# An INNER join would silently drop it, hiding a potential data quality issue.
```
</details>

## 8. Raster basics — vectors vs. rasters

Everything so far (points, lines, polygons) is **vector** data — discrete shapes with exact coordinates. A **raster** is fundamentally different: a grid of cells (pixels), each holding a value, covering continuous space. WorldPop population data is a raster: instead of "here is a polygon with population 328,900," it's "here's a grid where each 100m×100m cell has its own population estimate."

Key raster concepts:
- **Resolution** — the size of each grid cell (e.g., WorldPop is often 100m resolution — each pixel represents a 100m × 100m area)
- **Band** — a single layer of values (population rasters usually have 1 band; satellite imagery often has multiple, e.g. red/green/blue)
- **Extracting values at a point/polygon** — "zonal statistics," e.g. "sum all raster cells that fall within this sub-city polygon" — this is exactly what replacing your synthetic population grid with real WorldPop data requires (`rasterstats` or `rasterio` + `numpy` handle this)

This is conceptual for now — the hands-on raster work comes when you actually swap in WorldPop data for your project, which is a Phase 7 task.

### Exercise 8.1 (conceptual — no code)
In your own words (write in a markdown cell or comment), explain: why would a 100m-resolution population raster potentially misrepresent the population of a very small, dense sub-city like Kirkos (only 14.6 km²)? Think about what happens at the edges of an irregularly-shaped boundary when it's approximated by square grid cells.

In [ ]:
# Write your answer as a comment or in a new markdown cell


<details>
<summary>Discussion (not a strict "solution" — a conceptual answer)</summary>

A 100m grid cell is 1 hectare (10,000 m²). For a sub-city as small as Kirkos (14.6 km² = 1,460 hectares), any grid cell that straddles the boundary gets counted as either fully inside or requires partial-area weighting — done naively, this can meaningfully over- or under-count population near edges, since real boundaries are irregular polygons, not neat grid-aligned shapes. The smaller and more irregular the polygon relative to the raster resolution, the bigger this edge effect becomes. This is exactly why "zonal statistics" tools use partial-cell weighting (fraction of each edge cell inside the polygon) rather than simple inside/outside cutoffs, for better accuracy on small areas.
</details>

## 9. Checkpoint — rebuild your project's core accessibility logic

This mirrors exactly what your `addis-transit-access` project does, just simplified. Do this **without opening your actual project repo** — the goal is proving you understand the pipeline, not copying it.

**Task:**
1. Create a `GeoDataFrame` of at least 3 sub-city polygons with `sub_city` and `population` columns (reuse/adapt `subcity_polygons` from Section 3, or make your own), in `EPSG:4326`
2. Create a `GeoDataFrame` of at least 5 transit stop points scattered across/near those polygons, in `EPSG:4326`
3. Reproject **both** to `EPSG:32637`
4. Buffer each stop by 500 meters
5. For each sub-city polygon, calculate what fraction of its area is covered by the **unioned** stop buffers (i.e., "what percentage of this sub-city is within 500m of a transit stop?") — this is your accessibility metric
6. Print sub-cities sorted by this coverage percentage, lowest first (most underserved)

Tip for step 5: `polygon.intersection(buffer_union).area / polygon.area * 100` gives the percentage covered.

In [12]:
# Your checkpoint code here
# 1. Sub-city polygons
subcities_check = gpd.GeoDataFrame({
    "sub_city": ["Bole", "Yeka", "Kirkos"],
    "population": [328900, 401000, 235100],
    "geometry": [
        Polygon([(38.74, 8.98), (38.82, 8.98), (38.82, 9.05), (38.74, 9.05)]),
        Polygon([(38.80, 9.00), (38.88, 9.00), (38.88, 9.08), (38.80, 9.08)]),
        Polygon([(38.72, 8.96), (38.76, 8.96), (38.76, 9.00), (38.72, 9.00)])
    ]
}, crs="EPSG:4326")

# 2. Transit stops (deliberately sparse near Kirkos to show low coverage)
stops_check = gpd.GeoDataFrame({
    "stop_name": ["S1", "S2", "S3", "S4", "S5"],
    "geometry": [
        Point(38.76, 9.00), Point(38.78, 9.02), Point(38.80, 9.01),
        Point(38.84, 9.04), Point(38.86, 9.06)
    ]
}, crs="EPSG:4326")

# 3. Reproject both
subcities_utm = subcities_check.to_crs("EPSG:32637")
stops_utm = stops_check.to_crs("EPSG:32637")

# 4. Buffer stops
stop_buffers = stops_utm.geometry.buffer(500)
buffer_union = stop_buffers.union_all()   # merge all buffers into one shape

# 5. Coverage percentage per sub-city
coverage = []
for _, row in subcities_utm.iterrows():
    intersection_area = row.geometry.intersection(buffer_union).area
    pct_covered = (intersection_area / row.geometry.area) * 100
    coverage.append(pct_covered)

subcities_utm["coverage_pct"] = coverage

# 6. Sort — most underserved first
result = subcities_utm.sort_values("coverage_pct")[["sub_city", "population", "coverage_pct"]]
print(result)

  sub_city  population  coverage_pct
2   Kirkos      235100      1.008220
1     Yeka      401000      2.520896
0     Bole      328900      3.456976


<details>
<summary>Reference solution</summary>

```python
# 1. Sub-city polygons
subcities_check = gpd.GeoDataFrame({
    "sub_city": ["Bole", "Yeka", "Kirkos"],
    "population": [328900, 401000, 235100],
    "geometry": [
        Polygon([(38.74, 8.98), (38.82, 8.98), (38.82, 9.05), (38.74, 9.05)]),
        Polygon([(38.80, 9.00), (38.88, 9.00), (38.88, 9.08), (38.80, 9.08)]),
        Polygon([(38.72, 8.96), (38.76, 8.96), (38.76, 9.00), (38.72, 9.00)])
    ]
}, crs="EPSG:4326")

# 2. Transit stops (deliberately sparse near Kirkos to show low coverage)
stops_check = gpd.GeoDataFrame({
    "stop_name": ["S1", "S2", "S3", "S4", "S5"],
    "geometry": [
        Point(38.76, 9.00), Point(38.78, 9.02), Point(38.80, 9.01),
        Point(38.84, 9.04), Point(38.86, 9.06)
    ]
}, crs="EPSG:4326")

# 3. Reproject both
subcities_utm = subcities_check.to_crs("EPSG:32637")
stops_utm = stops_check.to_crs("EPSG:32637")

# 4. Buffer stops
stop_buffers = stops_utm.geometry.buffer(500)
buffer_union = stop_buffers.union_all()   # merge all buffers into one shape

# 5. Coverage percentage per sub-city
coverage = []
for _, row in subcities_utm.iterrows():
    intersection_area = row.geometry.intersection(buffer_union).area
    pct_covered = (intersection_area / row.geometry.area) * 100
    coverage.append(pct_covered)

subcities_utm["coverage_pct"] = coverage

# 6. Sort — most underserved first
result = subcities_utm.sort_values("coverage_pct")[["sub_city", "population", "coverage_pct"]]
print(result)
```

Note: `.union_all()` is the current GeoPandas method name (older code/tutorials may show `.unary_union` — same idea, GeoPandas renamed it).
</details>

## Next steps

With Phase 4 solid, the concepts underneath your entire `addis-transit-access` project are now yours, not borrowed. From here you can either:
- Continue **Phase 5** (QGIS) in parallel, recreating this same buffer/intersect workflow visually
- Jump to **Phase 7** and actually swap real WorldPop raster data into your project, now that Section 8's raster concepts make that task legible instead of a black box
